# Implementación de modelos ARIMA, SARIMA, ARIMAX y SARIMAX para dengue

## 1. ¿Las métricas y evaluaciones propuestas están bien?

Sí. La estrategia metodológica que planteaste está bien estructurada y es coherente con trabajos de modelado de series temporales epidemiológicas.

Tu propuesta combina dos enfoques importantes:

1. Evaluación estadística del ajuste del modelo.
2. Evaluación predictiva del desempeño.

Esto es correcto porque un modelo puede ajustarse bien a los datos históricos, pero no necesariamente predecir bien.

---


## 2. Evaluaciones que sí debes aplicar

### A. Diagnóstico de residuos

#### Prueba de Ljung-Box

Permite verificar si los residuos del modelo son independientes.

Interpretación:

* p > 0.05 → los residuos parecen ruido blanco → buen ajuste.
* p < 0.05 → aún existe autocorrelación → el modelo no capturó toda la estructura temporal.

Esto es obligatorio en modelos ARIMA/SARIMA.

---

### B. Criterios de información

#### AIC

Penaliza modelos muy complejos.

Menor AIC = mejor equilibrio entre ajuste y complejidad.

#### BIC

Similar al AIC, pero castiga más los modelos con demasiados parámetros.

Menor BIC = modelo más parsimonioso.

---

### C. Métricas predictivas

#### RMSE

Mide el error promedio cuadrático.

Más sensible a errores grandes.

#### MAE

Mide el error absoluto promedio.

Más robusto ante valores extremos.

#### MAPE

Mide error porcentual.

Muy útil para interpretar el error relativo.

Interpretación aproximada:

* <10% → excelente.
* 10–20% → bueno.
* 20–50% → aceptable.
* > 50% → deficiente.

---

## 3. Sobre las métricas de clasificación binaria

También está correcto incluirlas, PERO:

NO deben aplicarse todavía en esta etapa.

Primero debes:

1. Construir los modelos ARIMA/SARIMA/ARIMAX/SARIMAX.
2. Generar predicciones continuas.
3. Definir un umbral epidemiológico.
4. Convertir las predicciones en:

* Epidemia = 1
* No epidemia = 0

Y SOLO después aplicar:

* Accuracy
* Recall
* F1-score
* ROC-AUC

Entonces metodológicamente estás haciendo bien en dejar esa parte para después.

---

# 4. Estructura del modelado

Según lo que comentaste:

* La serie se volvió estacionaria con d = 1.
* ACF y PACF sugieren un rezago principal.
* Existe estacionalidad semanal anual de 52 semanas.

Eso significa que el modelo base inicial sí puede ser:

## Modelo base

### ARIMA(1,1,1)

Pero metodológicamente NO es recomendable evaluar un único modelo.

En series temporales epidemiológicas lo correcto es construir varios modelos candidatos y posteriormente seleccionar el mejor según:

* AIC
* BIC
* RMSE
* MAE
* MAPE
* Diagnóstico de residuos

---

# 5. Modelos candidatos recomendados

## A. Modelos ARIMA candidatos

Como d = 1 ya está definido, debes variar principalmente:

* p
* q

Modelos recomendados:

| Modelo       | Justificación          |
| ------------ | ---------------------- |
| ARIMA(1,1,0) | Dominancia AR          |
| ARIMA(0,1,1) | Dominancia MA          |
| ARIMA(1,1,1) | Modelo base clásico    |
| ARIMA(2,1,1) | Mayor memoria temporal |
| ARIMA(1,1,2) | Mayor suavización      |
| ARIMA(2,1,2) | Modelo más flexible    |

---

## B. Modelos SARIMA candidatos

Como detectaste estacionalidad semanal anual:

s = 52

Entonces los modelos estacionales recomendados son:

| Modelo                  |
| ----------------------- |
| SARIMA(1,1,1)(1,1,1,52) |
| SARIMA(1,1,0)(1,1,1,52) |
| SARIMA(0,1,1)(1,1,1,52) |
| SARIMA(2,1,1)(1,1,1,52) |
| SARIMA(1,1,2)(1,1,1,52) |
| SARIMA(1,1,1)(0,1,1,52) |
| SARIMA(1,1,1)(1,1,0,52) |

Esto es importante porque:

* algunas veces la dinámica no está en el componente regular;
* otras veces está principalmente en la parte estacional.

---

## C. Modelos ARIMAX candidatos

Los mismos órdenes ARIMA, pero incluyendo variables climáticas:

| Modelo        |
| ------------- |
| ARIMAX(1,1,0) |
| ARIMAX(0,1,1) |
| ARIMAX(1,1,1) |
| ARIMAX(2,1,1) |
| ARIMAX(1,1,2) |

---

## D. Modelos SARIMAX candidatos

| Modelo                   |
| ------------------------ |
| SARIMAX(1,1,1)(1,1,1,52) |
| SARIMAX(1,1,0)(1,1,1,52) |
| SARIMAX(0,1,1)(1,1,1,52) |
| SARIMAX(2,1,1)(1,1,1,52) |
| SARIMAX(1,1,2)(1,1,1,52) |

---

# 6. Estrategia metodológica recomendada

La mejor estrategia para tu tesis sería:

## Fase 1

Construir todos los modelos candidatos.

---

## Fase 2

Comparar:

* AIC
* BIC
* RMSE
* MAE
* MAPE
* Ljung-Box

---

## Fase 3

Seleccionar los mejores modelos:

* mejor ARIMA;
* mejor SARIMA;
* mejor ARIMAX;
* mejor SARIMAX.

---

## Fase 4

Comparar entre familias.

Por ejemplo:

* ¿SARIMAX mejora frente a ARIMA?
* ¿las variables climáticas mejoran la predicción?
* ¿la estacionalidad mejora el desempeño?

Eso fortalece muchísimo la discusión científica.

---

# 7. Variables exógenas disponibles

En esta base de datos se observan las siguientes variables:

* hum_esp_bc_lag_1	
* vel_vi_max_bc_lag_1	
* vel_vi_bc_lag_4	
* dias_lluvia_ln_lag_5	
* temp_max_bc_lag_1	
* sst_yj_lag_10


Variable objetivo:

* casos_ln

Fecha:

* fecha

---



# 1. Librerías 

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf

from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.stats.diagnostic import acorr_ljungbox

import warnings
warnings.filterwarnings("ignore")

# 2. Carga de datos

In [22]:
df_corr = pd.read_excel(r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Modelos\modelo_corr_rezago_relevante\datos_corr_rezago_relevante.xlsx")

df_corr['fecha'] = pd.to_datetime(df_corr['fecha'])
df_corr = df_corr.sort_values('fecha')
df_corr.set_index('fecha', inplace=True)
df_corr.columns

Index(['año', 'semana_epi', 'casos_ln', 'hum_esp_bc_lag_1',
       'vel_vi_max_bc_lag_1', 'vel_vi_bc_lag_4', 'dias_lluvia_ln_lag_5',
       'temp_max_bc_lag_1', 'sst_yj_lag_10'],
      dtype='object')

# 3. Rutas

In [12]:
ruta_resultados = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Modelos\modelo_corr_rezago_relevante\Resultados"

ruta_graficos = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Modelos\modelo_corr_rezago_relevante\Gráficos"

os.makedirs(ruta_resultados, exist_ok=True)
os.makedirs(ruta_graficos, exist_ok=True)

# 4. Variables

In [5]:
y = df_corr['casos_ln']
exog = df_corr.drop(columns=['casos_ln'])

# 5. Funciones base

##  --Métricas--

In [7]:
def metricas(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return rmse, mae, mape

# --Ljung-Box--

In [8]:
def ljung_box(residuos):
    return acorr_ljungbox(residuos, lags=[10], return_df=True)['lb_pvalue'].values[0]

# --Split--

In [9]:
def split(y, exog=None, size=0.8):
    n = int(len(y) * size)
    y_train, y_test = y[:n], y[n:]
    
    if exog is not None:
        return y_train, y_test, exog[:n], exog[n:]
    
    return y_train, y_test

# 6. Función principal de modelos

In [10]:
def correr_modelo(y_train, y_test, order, seasonal=None, exog_train=None, exog_test=None):
    
    try:
        if seasonal:
            model = SARIMAX(y_train,
                            order=order,
                            seasonal_order=seasonal,
                            exog=exog_train,
                            enforce_stationarity=False,
                            enforce_invertibility=False)
        else:
            model = ARIMA(y_train, order=order, exog=exog_train)
        
        fit = model.fit()
        pred = fit.forecast(len(y_test), exog=exog_test)
        
        rmse, mae, mape = metricas(y_test, pred)
        lb = ljung_box(fit.resid)
        
        return fit, pred, {
            "AIC": fit.aic,
            "BIC": fit.bic,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape,
            "Ljung-Box": lb
        }
    
    except:
        return None, None, None

# 7. Funciones de gráficos

## --Predicción--

In [17]:
def plot_pred(y_train, y_test, pred, nombre):
    plt.figure(figsize=(12,6))
    plt.plot(y_train, label='Train')
    plt.plot(y_test, label='Test')
    plt.plot(y_test.index, pred, label='Predicción')
    plt.legend()
    plt.title(nombre)
    plt.savefig(os.path.join(ruta_graficos, f"{nombre}_pred.png"))
    plt.close()

## --Diagnóstico residuos--

In [13]:
def plot_residuos(residuos, nombre):
    
    plt.figure(figsize=(12,4))
    plt.plot(residuos)
    plt.title(f"Residuos - {nombre}")
    plt.savefig(os.path.join(ruta_graficos, f"{nombre}_residuos.png"))
    plt.close()
    
    plot_acf(residuos, lags=30)
    plt.title(f"ACF Residuos - {nombre}")
    plt.savefig(os.path.join(ruta_graficos, f"{nombre}_acf.png"))
    plt.close()
    
    plt.figure()
    plt.hist(residuos, bins=20)
    plt.title(f"Histograma Residuos - {nombre}")
    plt.savefig(os.path.join(ruta_graficos, f"{nombre}_hist.png"))
    plt.close()

# 8. Modelos candidatos

In [14]:
arima_orders = [(1,1,0),(0,1,1),(1,1,1),(2,1,1),(1,1,2),(2,1,2)]

sarima_orders = [
((1,1,1),(1,1,1,52)),
((1,1,0),(1,1,1,52)),
((0,1,1),(1,1,1,52)),
((2,1,1),(1,1,1,52)),
((1,1,2),(1,1,1,52))
]

# 9. Función general de ejecución

In [15]:
def ejecutar_modelos(nombre_familia, orders, usar_exog=False, usar_seasonal=False, split_size=0.8):
    
    resultados = []
    
    if usar_exog:
        y_train, y_test, ex_train, ex_test = split(y, exog, split_size)
    else:
        y_train, y_test = split(y, size=split_size)
        ex_train, ex_test = None, None
    
    for item in orders:
        
        if usar_seasonal:
            order, seasonal = item
        else:
            order, seasonal = item, None
        
        nombre = f"{nombre_familia}_{order}"
        
        fit, pred, metrics = correr_modelo(
            y_train, y_test,
            order,
            seasonal,
            ex_train,
            ex_test
        )
        
        if metrics:
            plot_pred(y_train, y_test, pred, nombre)
            plot_residuos(fit.resid, nombre)
            
            metrics["Modelo"] = nombre
            resultados.append(metrics)
    
    df_res = pd.DataFrame(resultados).sort_values("AIC")
    
    df_res.to_excel(os.path.join(ruta_resultados, f"{nombre_familia}_{int(split_size*100)}.xlsx"), index=False)
    
    return df_res

# 10. Ejecución total

## --80-20--

In [18]:
res_arima_80 = ejecutar_modelos("ARIMA", arima_orders, False, False, 0.8)
res_sarima_80 = ejecutar_modelos("SARIMA", sarima_orders, False, True, 0.8)
res_arimax_80 = ejecutar_modelos("ARIMAX", arima_orders, True, False, 0.8)
res_sarimax_80 = ejecutar_modelos("SARIMAX", sarima_orders, True, True, 0.8)

## --90-10--

In [19]:
res_arima_90 = ejecutar_modelos("ARIMA", arima_orders, False, False, 0.9)
res_sarima_90 = ejecutar_modelos("SARIMA", sarima_orders, False, True, 0.9)
res_arimax_90 = ejecutar_modelos("ARIMAX", arima_orders, True, False, 0.9)
res_sarimax_90 = ejecutar_modelos("SARIMAX", sarima_orders, True, True, 0.9)

# 11. Mejor modelo automático

In [20]:
print("Mejor ARIMA 80:\n", res_arima_80.iloc[0])
print("Mejor SARIMA 80:\n", res_sarima_80.iloc[0])
print("Mejor ARIMAX 80:\n", res_arimax_80.iloc[0])
print("Mejor SARIMAX 80:\n", res_sarimax_80.iloc[0])

Mejor ARIMA 80:
 AIC               124.596052
BIC               131.172586
RMSE                0.902193
MAE                 0.706996
MAPE              235.586319
Ljung-Box           0.497801
Modelo       ARIMA_(0, 1, 1)
Name: 1, dtype: object
Mejor SARIMA 80:
 AIC                 67.253129
BIC                  82.38386
RMSE                 1.002221
MAE                  0.757056
MAPE               240.541445
Ljung-Box            0.006211
Modelo       SARIMA_(2, 1, 1)
Name: 3, dtype: object
Mejor ARIMAX 80:
 AIC                120.074573
BIC                 156.24551
RMSE                 1.227219
MAE                  0.966693
MAPE               320.167291
Ljung-Box                 1.0
Modelo       ARIMAX_(1, 1, 1)
Name: 2, dtype: object
Mejor SARIMAX 80:
 AIC                  84.541227
BIC                  119.69326
RMSE                  0.973314
MAE                    0.76156
MAPE                235.436905
Ljung-Box             0.134995
Modelo       SARIMAX_(1, 1, 2)
Name: 4, dtype: obj

In [21]:
print("Mejor ARIMA 90:\n", res_arima_90.iloc[0])
print("Mejor SARIMA 90:\n", res_sarima_90.iloc[0])
print("Mejor ARIMAX 90:\n", res_arimax_90.iloc[0])
print("Mejor SARIMAX 90:\n", res_sarimax_90.iloc[0])

Mejor ARIMA 90:
 AIC               121.931663
BIC               128.746007
RMSE                0.770687
MAE                 0.612804
MAPE              270.027891
Ljung-Box           0.536083
Modelo       ARIMA_(0, 1, 1)
Name: 1, dtype: object
Mejor SARIMA 90:
 AIC               3425.661968
BIC               3436.710664
RMSE                 0.815453
MAE                  0.540636
MAPE               222.779696
Ljung-Box            0.497929
Modelo       SARIMA_(0, 1, 1)
Name: 2, dtype: object
Mejor ARIMAX 90:
 AIC                116.403791
BIC                157.289853
RMSE                  1.12278
MAE                  0.995964
MAPE               412.648063
Ljung-Box                 1.0
Modelo       ARIMAX_(2, 1, 1)
Name: 3, dtype: object
Mejor SARIMAX 90:
 AIC                 4594.80852
BIC                4633.358783
RMSE                  0.817892
MAE                   0.552792
MAPE                231.903376
Ljung-Box             0.703988
Modelo       SARIMAX_(1, 1, 2)
Name: 4, dtype: obj